# 11 — Observability & tracing: watching your program run

**What you'll learn**

- The hook layer every tracing tool stands on: `dspy.utils.callback.BaseCallback` — and how to build a working tracer with it in ~20 lines.
- MLflow, DSPy's first-party observability tool: automatic trace trees for every call, and — the headline — **optimizer compiles recorded as parent/child runs** you can browse. The UI launches from a notebook cell; you never open a terminal.
- Option B: **Arize Phoenix**, a purpose-built LLM-trace UI, and when to prefer it.
- What those commented lines at the bottom of every config cell in this course actually do.

*Time: ~30 min. Cost: ~$0.05 — and near-$0 if you ran notebooks 04–05 on this machine, because almost every LM call below is a disk-cache hit.*

In [ ]:
# === config (identical in every notebook) ===
import os, getpass
import dspy
from dotenv import load_dotenv              # pip install -r requirements.txt if this fails

load_dotenv(".env")   # reads OPENROUTER_API_KEY / MODEL / STRONG_MODEL (see .env.example)

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OpenRouter API key: ")

MODEL = os.environ.get("MODEL", "openrouter/deepseek/deepseek-v3.2")
STRONG_MODEL = os.environ.get("STRONG_MODEL", "openrouter/deepseek/deepseek-v4-flash")

# Per-notebook override: uncomment to ignore .env here (any LiteLLM provider works).
# MODEL = "openrouter/google/gemini-2.5-flash-lite"
# MODEL = "openai/gpt-4o-mini"              # direct OpenAI, uses OPENAI_API_KEY instead
# STRONG_MODEL = "openrouter/moonshotai/kimi-k2.6"
# STRONG_MODEL = "openrouter/openai/gpt-4o"

lm = dspy.LM(MODEL, max_tokens=1000)        # cache=True by default: reruns are ~free
dspy.configure(lm=lm)

# Optional tracing for any notebook (taught in notebook 11) — pick ONE option:
# Option A, MLflow: logs land in ./mlflow.db; notebook 11 launches the UI from a cell.
#   import mlflow
#   mlflow.set_experiment("dspy-lab")
#   mlflow.dspy.autolog()
# Option B, Arize Phoenix: needs the Phoenix server up (see notebook 11's launcher cell).
#   from phoenix.otel import register
#   register(project_name="dspy-lab", auto_instrument=True)

Boilerplate — same code as notebooks 03 and 04. Skim and run.

In [ ]:
# === boilerplate: aldervane micro-world — same code as notebook 03 ===
import json, math, re
from collections import Counter
from pathlib import Path

CORPUS = json.loads(Path("data/corpus.json").read_text())

STOPWORDS = {
    "a", "an", "and", "as", "at", "by", "did", "do", "does", "each", "every",
    "for", "from", "he", "her", "his", "in", "is", "it", "its", "of", "on",
    "or", "she", "that", "the", "this", "to", "was", "were", "what", "when",
    "where", "which", "who", "whom", "whose", "with",
}

def tokenize(text):
    words = re.findall(r"[a-z0-9]+", text.lower())
    # naive stemming: "captains" -> "captain", "records" -> "record"
    words = [w[:-1] if len(w) > 3 and w.endswith("s") and not w.endswith("ss") else w
             for w in words]
    return [w for w in words if w not in STOPWORDS]

DOC_TOKENS = {p["id"]: set(tokenize(p["title"] + " " + p["text"])) for p in CORPUS}
IDF = Counter()
for tokens in DOC_TOKENS.values():
    IDF.update(tokens)
IDF = {t: math.log(len(CORPUS) / df) for t, df in IDF.items()}

def search(query, k=2):
    """Return the k corpus passages that best match the query (keyword + IDF)."""
    q = set(tokenize(query))
    ranked = sorted(
        CORPUS,
        key=lambda p: (-sum(IDF.get(t, 0.0) for t in q & DOC_TOKENS[p["id"]]), p["id"]),
    )
    return ranked[:k]

def as_context(passages):
    return [f"{p['title']}: {p['text']}" for p in passages]

class HopQA(dspy.Module):
    """Two-hop retrieval QA over the Aldervane corpus."""

    def __init__(self, k=2):
        super().__init__()
        self.k = k
        self.hop1 = dspy.Predict("question -> search_query")
        self.hop2 = dspy.Predict("question, context: list[str] -> search_query")
        self.answer = dspy.ChainOfThought("question, context: list[str] -> answer")

    def forward(self, question):
        q1 = self.hop1(question=question).search_query
        first = search(q1, k=self.k)
        q2 = self.hop2(question=question, context=as_context(first)).search_query
        second = search(q2, k=self.k)
        passages = {p["id"]: p for p in first + second}      # merge + dedupe
        pred = self.answer(question=question, context=as_context(list(passages.values())))
        return dspy.Prediction(
            answer=pred.answer,
            reasoning=pred.reasoning,
            queries=[q1, q2],
            retrieved_ids=sorted(passages),
        )

In [ ]:
# === boilerplate: data + metric — same code as notebook 04 ===
QA = json.loads(Path("data/qa.json").read_text())

def to_examples(rows):
    return [dspy.Example(**row).with_inputs("question") for row in rows]

train, dev, test = (to_examples(QA[s]) for s in ("train", "dev", "test"))

def norm(s):
    s = re.sub(r"[^a-z0-9\s]", " ", s.lower())
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())

def em_metric(example, pred, trace=None):
    """1.0 if the predicted answer exactly matches the gold answer (after normalization)."""
    return norm(pred.answer) == norm(example.answer)

## The question this notebook answers

Ten notebooks taught you to *improve* a program. Production asks a different question: **what is my program doing right now?** Which module is slow, which call failed to parse, what did the retriever actually return at 3 a.m. for that one angry user?

You have been living on `dspy.inspect_history()` — a fine reflex (notebook 01), but look at its limits with fresh eyes:

1. **It only logs LM calls.** Our `search()` node — half the program's behavior — is invisible to it. Tools, retrievers, custom Python: all dark.
2. **It is one monolithic transcript.** After an `Evaluate` over 16 questions, "the last n calls" interleaves everything; good luck telling hop-2-of-question-7 from hop-1-of-question-8.
3. **No metadata.** No latency, no token counts, no parent/child structure — you saw in notebook 03 that one program call ≠ one LM call, but inspect_history flattens that tree away.

Notice something: in notebook 04 you *hand-rolled* observability — we made `forward` return `queries` and `retrieved_ids` precisely so failures could be diagnosed. That instinct was right. This notebook industrializes it.

## The hook layer: `BaseCallback`

DSPy exposes the seams of every execution as a callback API. Subclass `dspy.utils.callback.BaseCallback`, implement any of twelve hooks, register with `dspy.configure(callbacks=[...])`, and DSPy calls you at each boundary:

| Hook pair | Fires when |
|---|---|
| `on_module_start` / `on_module_end` | any `dspy.Module` is invoked (including `Predict` itself) |
| `on_lm_start` / `on_lm_end` | a `dspy.LM` is invoked |
| `on_adapter_format_start` / `..._end` | an adapter renders the prompt (notebook 02's `format`) |
| `on_adapter_parse_start` / `..._end` | an adapter parses the reply |
| `on_tool_start` / `on_tool_end` | a `dspy.Tool` is invoked |
| `on_evaluate_start` / `on_evaluate_end` | a `dspy.Evaluate` runs |

Two details that matter when you implement one:

- **The signatures are asymmetric.** `*_start(call_id, instance, inputs)` but `*_end(call_id, outputs, exception)` — the end hook does *not* get the instance. The `call_id` is your correlation key; keep a dict.
- **Nesting is tracked for you.** DSPy maintains the active call context (a `ContextVar`), which is how a tracer knows that this LM call happened *inside* that `Predict` *inside* your `HopQA`. That context is the span tree.

One boundary to know: hooks fire for DSPy objects. A plain Python function like our `search()` never triggers anything — only a `dspy.Tool` does (that gap becomes exercise 2).

In [ ]:
import time
from dspy.utils.callback import BaseCallback

class MiniTracer(BaseCallback):
    """Print an indented span tree of module and LM calls, with latency."""

    def __init__(self):
        self.depth = 0
        self.started = {}

    def _open(self, call_id, label):
        print("  " * self.depth + f"> {label}")
        self.started[call_id] = time.perf_counter()
        self.depth += 1

    def _close(self, call_id):
        self.depth -= 1
        ms = 1000 * (time.perf_counter() - self.started.pop(call_id, time.perf_counter()))
        print("  " * self.depth + f"< {ms:.0f} ms")

    def on_module_start(self, call_id, instance, inputs):
        self._open(call_id, type(instance).__name__)

    def on_module_end(self, call_id, outputs, exception=None):
        self._close(call_id)

    def on_lm_start(self, call_id, instance, inputs):
        self._open(call_id, f"LM ({instance.model})")

    def on_lm_end(self, call_id, outputs, exception=None):
        self._close(call_id)

dspy.configure(callbacks=[MiniTracer()])
pred = HopQA()(question=dev[1].question)
dspy.configure(callbacks=[])       # unhook — stop tracing before the eval/compile cells below

print("\nanswer:", pred.answer)

> **What you should see:** an indented tree, something like:
>
> ```
> > HopQA
>   > Predict
>     > LM (openrouter/deepseek/deepseek-v3.2)
>     < 4 ms
>   < 6 ms
>   > Predict
>     ...
>   > ChainOfThought
>     > Predict
>       > LM (...)
> ```
>
> with near-zero millisecond numbers if the calls were cache hits. Three predictors, their LM calls, correctly nested — the structure notebook 03 could only *tell* you about, now printed live. Note what is present and absent: `Predict`/`ChainOfThought` appear (they are Modules), `search()` does not (plain function).
>
> `inspect_history` shows *content* (the full prompts); a tracer shows *structure and timing*. You want both, and they are complements — not rivals.
>
> One more thing before we upgrade tools: the class you just wrote is not a toy pattern. Braintrust's first-party DSPy integration is built on this very `BaseCallback` API. The hook layer you now understand is what commercial observability stands on.

## The industrial version: MLflow

MLflow is DSPy's first-party observability tool — local, open-source, no signup, no API key (it is already in this course's `requirements.txt`; the floor for what we do below is `mlflow>=2.21.1`). Three facts before the code:

- **Logging needs no server.** Every trace below is written to one SQLite file, `mlflow.db`, at the repo root (gitignored; older MLflow versions used an `./mlruns` directory instead). The next cell pins that path *explicitly*, so the logger and the UI can never disagree about where the logs live — no matter where anything was launched from.
- **The UI never makes you leave the notebook.** A cell below starts it as a background process and prints the link; the only thing you do outside Jupyter is click.
- **Everything can happen later.** Skip the UI cell now and run it tomorrow — the store is on disk; nothing is lost.

In [ ]:
# This is the config cell's commented "Option A", done live — plus a pinned store path.
# (Leave the frozen config cell untouched — it stays identical across notebooks.)
import mlflow

MLFLOW_DB = Path("mlflow.db").resolve()      # ./mlflow.db at the repo root (gitignored)
mlflow.set_tracking_uri(f"sqlite:///{MLFLOW_DB}")
mlflow.set_experiment("dspy-lab")
mlflow.dspy.autolog()              # defaults: trace inference (and evals); don't log runs yet

In [ ]:
# OPTIONAL — start the MLflow UI right here; no terminal needed. Then click the link.
import socket, subprocess, sys, time

def launch_ui(module_args, port, name):
    """Start a local web UI as a background process; if the port already answers, reuse it."""
    with socket.socket() as probe:
        if probe.connect_ex(("127.0.0.1", port)) == 0:
            print(f"{name} already running: http://127.0.0.1:{port}")
            return
    subprocess.Popen([sys.executable, "-m", *module_args],
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(120):                                 # first start can take ~10 s
        with socket.socket() as probe:
            if probe.connect_ex(("127.0.0.1", port)) == 0:
                print(f"{name}: http://127.0.0.1:{port}")
                return
        time.sleep(0.5)
    print(f"{name} did not answer on port {port}; try another port, or run it from a terminal.")

launch_ui(["mlflow", "ui", "--backend-store-uri", f"sqlite:///{MLFLOW_DB}", "--port", "5000"],
          port=5000, name="MLflow UI")

> **What you should see:** `MLflow UI: http://127.0.0.1:5000` after a few seconds — click it. Two properties worth noticing, both bought by ~15 lines of plain Python: the UI is a background *process*, not part of this kernel — restarting the kernel leaves it up, and rerunning the cell just finds it and reprints the link (that is what the port probe is for). And because we pass the pinned `MLFLOW_DB` to the server, it is reading exactly the store the cells above write — the "launched the UI in the wrong directory, everything looks empty" failure mode is engineered away. (To stop it eventually: kill whatever holds port 5000, or just leave it — it idles.)

In [ ]:
evaluate = dspy.Evaluate(devset=dev, metric=em_metric, num_threads=8, display_progress=True)

pred = HopQA()(question=dev[2].question)
result = evaluate(HopQA())
result.score

> **What you should see — in the notebook:** nothing new. Tracing is passive; outputs are unchanged. That silence is the point — observability that costs you no code changes.
>
> **In the UI** (the launcher's link): experiment `dspy-lab` → **Traces** tab → one trace per program call, each expanding into the same tree MiniTracer printed — module > predict > adapter format > LM > adapter parse — but now with inputs/outputs per span, token counts, and latencies. The evaluation contributed 16 more traces (eval tracing is on by default). Cache-hit spans show near-zero latency: visible proof of what the disk cache has been saving you all course.

## The knobs — and the headline feature

`mlflow.dspy.autolog` has seven switches; the defaults trace inference but stay quiet about training:

| knob | default | meaning |
|---|---|---|
| `log_traces` | `True` | trace ordinary program calls |
| `log_traces_from_eval` | `True` | trace calls made inside `dspy.Evaluate` |
| `log_traces_from_compile` | `False` | trace every call inside `optimizer.compile` (floods — see below) |
| `log_compiles` | `False` | record each compile as a **run** (params, candidates, artifacts) |
| `log_evals` | `False` | record each evaluation as a run |
| `disable` | `False` | `autolog(disable=True)` switches tracing off again |
| `silent` | `False` | suppress MLflow's own log lines |

The headline is `log_compiles`. Notebooks 05–07 kept asking you to *trust* that an optimizer tried candidates and kept the best; MLflow records the receipts: **one parent run per compile, one child run per candidate program**, each with its parameters, its prompts, and its score.

*(Scaling note, not needed today: the launcher cell above already runs a real MLflow server on your pinned store — that is all `mlflow ui` is. For a shared or remote box, run it from a terminal instead — `mlflow server --backend-store-uri sqlite:///mydb.sqlite --host 0.0.0.0` — and point notebooks at it with `mlflow.set_tracking_uri("http://<host>:5000")`.)*

In [ ]:
mlflow.dspy.autolog(log_compiles=True, log_evals=True)   # log_traces_from_compile stays False: no flood

# Identical to notebook 05's random-search compile — so the LM calls are disk-cached and ~free,
# while MLflow still records the full optimization (it hooks the optimizer, not the network).
rs = dspy.BootstrapFewShotWithRandomSearch(
    metric=em_metric,
    max_bootstrapped_demos=4,
    max_labeled_demos=4,
    num_candidate_programs=4,
    num_threads=8,
)
boot_rs = rs.compile(HopQA(), trainset=train, valset=dev)

evaluate(boot_rs).score

> **What you should see — in the notebook:** the familiar notebook-05 compile logs (fast, thanks to the cache), then a dev score matching what you got there.
>
> **In the UI:** the Runs view of `dspy-lab` now shows a **parent run** for the compile with **child runs** underneath — the candidate programs random search actually built — each carrying its parameters and score, with the winning program's JSON saved as an artifact (the same `save()` format you have been reading since notebook 02). The final evaluation appears as its own run too (`log_evals=True`). This is the course's "read the learned parameters" ritual, with a database and a UI behind it.
>
> Why we used the random-search optimizer for the demo: a *single-shot* optimizer like plain `BootstrapFewShot` builds one program in one pass — its compile run has no candidate tree. The parent/child story needs an optimizer that searches.
>
> Two footnotes: GEPA (notebook 07) also has its own built-in opt-in — `dspy.GEPA(use_mlflow=True, mlflow_tracking_uri=..., mlflow_experiment_name=...)` — which logs its evolutionary tree the same way. And you now know exactly what the commented block at the bottom of **every** config cell in this course does: uncomment it in any notebook to trace that notebook.

## Option B: Arize Phoenix — a purpose-built trace UI

MLflow is the course default because it is DSPy's first-party tool and the only one that understands *optimizer runs*. But for day-to-day **trace inspection** — staring at span waterfalls while you debug a `forward()` — many people prefer **Phoenix**: an open-source, fully local observability app whose UI was designed for LLM traces from the start (waterfall views, token/latency breakdowns per span, filtering, projects). It speaks OpenTelemetry, so the same instrumentation can later ship spans to any OTel backend.

The honest trade-off, stated once: **Phoenix has no notion of optimizer runs.** A `compile()` appears as a stream of individual traces — no parent/child candidates, no params, no artifacts. Hence the pairing this course recommends: *Phoenix while developing your program, MLflow when compiling it.*

Setup: both packages are in `requirements.txt`, and the `launch_ui` helper from the MLflow section works unchanged — one cell, no terminal (Phoenix's first start takes ~10–15 s). One difference from MLflow's file logging: **Phoenix receives spans over the network, so it must be up before your traced cells run** — which is exactly why the launch comes *first* in the next cell. Its data lives in `~/.phoenix` (outside the repo — nothing to gitignore), and the server, like MLflow's, outlives the kernel. *(Phoenix also ships an in-process launcher, `phoenix.launch_app()` — quicker, but its data is ephemeral by default and it dies with the kernel; we prefer the background server.)*

In [ ]:
mlflow.dspy.autolog(disable=True)        # one tracer at a time keeps the picture clean

launch_ui(["phoenix.server.main", "serve"], port=6006, name="Phoenix")

from phoenix.otel import register

tracer_provider = register(project_name="dspy-lab", auto_instrument=True)
# auto_instrument=True finds the installed openinference-instrumentation-dspy
# package and patches DSPy — no further wiring needed.

HopQA()(question=dev[3].question)

> **What you should see — in the notebook:** the launcher line (`Phoenix: http://127.0.0.1:6006`), a Phoenix banner from `register(...)`, and the usual `Prediction`. If the launcher instead printed its did-not-answer line (rare — heavily locked-down environments), fall back to `phoenix serve` in a terminal and rerun the cell; OpenTelemetry exporter warnings mean spans are being dropped for want of a live server.
>
> **In the UI** (http://localhost:6006): project `dspy-lab` → a trace whose waterfall shows the HopQA span with nested Predict/ChainOfThought and LM spans — prompts, completions, token counts and latency per span. Compare it side by side with MLflow's Traces tab rendering of the very same structure; which one you find more readable is a legitimate reason to choose (this option exists in the course precisely because Phoenix's UI tends to win that comparison).
>
> A caution for your own projects: Phoenix's DSPy *examples* online partly predate DSPy 3.x (`dspy.OpenAI`, `dspy.settings.configure` — patterns notebook 00 told you are gone). The instrumentation itself, as used above, is current; just don't copy their DSPy code.

## The wider landscape, and the choice

| Tool | Runs where | DSPy integration | Optimizer runs | Best at |
|---|---|---|---|---|
| **MLflow** (course primary) | local, free | first-party (in DSPy's own docs) | **yes** — parent/child compile runs | the full train-and-trace story |
| **Phoenix** (course Option B) | local, free | via `openinference-instrumentation-dspy` (OTel) | no | the nicest trace-inspection UI |
| Braintrust | SaaS (API key) | first-party, built on `BaseCallback` | partial (experiments-oriented) | teams already on its eval platform |
| LangSmith | SaaS (API key) | none — generic OTel ingestion | no | LangChain-ecosystem shops |

Per this course's discipline of mapping components to their sources: **no research paper backs this notebook — observability is engineering, not literature.** The primary sources are DSPy's own observability and optimizer-tracking tutorials at dspy.ai, and each tool's docs.

## Recap

| Concept | One-liner |
|---|---|
| `BaseCallback` | 12 hooks over modules / LMs / adapters / tools / evaluate; the layer tracers are built on |
| `dspy.configure(callbacks=[...])` | register (and `callbacks=[]` to unhook); correlate start/end via `call_id` |
| `mlflow.dspy.autolog()` | passive trace trees for every call; pinned SQLite store (`./mlflow.db`) |
| `launch_ui(...)` | start a local UI from a cell as a background process; port-probe makes it rerun-safe |
| `autolog(log_compiles=True, log_evals=True)` | compiles become parent/child candidate runs with params + program artifacts |
| `autolog(disable=True)` | tracing off again |
| Phoenix: `register(project_name=..., auto_instrument=True)` | OTel tracing into a purpose-built local UI (launch first — spans need a live collector) |
| the pairing | Phoenix while developing `forward()`; MLflow when compiling |
| GEPA | has its own `use_mlflow=True` opt-in |

## Exercises

1. In `MiniTracer.on_lm_end`, print `outputs` for a single call and go spelunking: where does your provider report token usage? Then extend the tracer to show tokens per LM span — you will have rebuilt the core of every cost dashboard.
2. Make `search()` visible: subclass `HopQA`, add `self.search_tool = dspy.Tool(search)` in `__init__`, and call `self.search_tool(query=...)` in `forward` instead of `search(...)`. Rerun the MiniTracer cell — `on_tool_start/end` spans appear — then rerun a traced call and find the tool span in MLflow or Phoenix, with the query and retrieved passages recorded.
3. *(Costs real money — notebook-06 prices.)* Rerun MIPROv2 `auto="light"` with `mlflow.dspy.autolog(log_compiles=True, log_traces_from_compile=True)` and find, in the UI, the candidate whose instructions won — then look at how many traces the compile produced, and reread the dedicated-server note above with new respect.

That's the course. Instrument something real.